# PSG backbone：组件单步调试
按顺序运行，每个 cell 只推进一个主要阶段。默认 CPU、小 batch；代码调用仓库实现，可以在模块 forward 中设置断点。QC 是来源 epoch 级，不在一秒切片上重新计算。

In [6]:
from pathlib import Path
import os
import sys
import torch
from dataclasses import replace

ROOT = Path.cwd()
if not (ROOT / "backbone").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
torch.set_num_threads(2)
torch.manual_seed(7)
DEVICE = os.getenv("PSG_NOTEBOOK_DEVICE", "cuda")
DATA_MODE_OVERRIDE = os.getenv("PSG_NOTEBOOK_REAL")
print(ROOT, DEVICE)

E:\Code\SleepWorldModel cpu


In [ ]:
from pretraining.configuration import load_config
from pretraining.factory import build_pretraining
from pretraining.runtime import autocast_context, build_optimizer, training_device
from dataloader import as_signal_batch, WindowDataset, collate_windows
from dataloader.synthetic import synthetic_windows

config = load_config(ROOT / "configs/psg_training.yaml", [
    "pretraining.sigreg.num_slices=8",  # 调试时减少随机投影数；正式默认1024
    "evaluation.max_batches=1",
])
if DATA_MODE_OVERRIDE is not None:
    config["data"]["mode"] = "real" if DATA_MODE_OVERRIDE == "1" else "synthetic"
if config["data"]["mode"] == "real":
    with WindowDataset(
        config["data"]["root"],
        context_epochs=1, night_grades=(3, 4, 5)
    ) as dataset:
        raw = collate_windows([dataset[0], dataset[1]])
else:
    config["evaluation"]["frozen_linear_probe"] = False
    raw = synthetic_windows(batch_size=2, epochs=1)
device = training_device(config["training"], DEVICE)
config["training"]["device"] = str(device)
batch = as_signal_batch(raw, scales=config["data"]["scales"], foundation=True)
model, view_sampler = build_pretraining(config, batch)
batch = batch.to(device, non_blocking=True)
model = model.to(DEVICE).eval()
backbone = model.backbone
print("night_grade", batch.night_grade)
print("parameters", sum(p.numel() for p in backbone.parameters()))
for name, group in batch.groups.items():
    print(name, group.values.shape, "epoch QC", group.valid.shape, group.channel_ids)

## 1. 原始波形 → 一秒 patch
选 EEG 作观察对象。PatchBatch.data_valid 保留 [B,C,E]，token_valid 是为了 attention 广播的派生有效性。

In [ ]:
encoder = backbone.modality_encoders["eeg"]
group = batch.groups["eeg"]
patches = encoder.patchifier(group, batch)
print("waveform", group.values.shape)
print("patch values", patches.values.shape)
print("epoch data_valid", patches.data_valid.shape)
print("derived token_valid", patches.token_valid.shape)
assert patches.data_valid.shape[-1] == 1

## 2. 三层 CNN
每次仅处理一个通道的一秒波形：[BCN,1,200] → [BCN,32,8] → 256。

In [ ]:
x = patches.values.reshape(-1, 1, 200)
for index, layer in enumerate(encoder.tokenizer.convolutions):
    x = layer(x)
    print(index, type(layer).__name__, tuple(x.shape))
flattened = x.flatten(1)
tokens = encoder.tokenizer(patches.values)
print("flatten", flattened.shape, "tokens", tokens.shape)
assert flattened.shape[-1] == 256

## 3. 单个 Criss-Cross block
valid 排除坏通道。通道分支和时间分支各128维；可展开 first_block 查看 Attention 与 FFN 参数。

In [ ]:
first_block = encoder.blocks[0]
active = patches.token_valid & patches.sample_visible.any(-1)
block_output = first_block(tokens, active)
print(first_block)
print("in/out", tokens.shape, block_output.shape)
assert not block_output[~active].any()

## 4. 调用完整模态编码器
返回 patch grid、局部编码 grid、通道池化结果以及池化权重。

In [ ]:
from backbone import MaskPlan

patch_grid, local_grid, eeg_features, channel_weights = encoder(group, batch, MaskPlan())
print("patch/local", patch_grid.tokens.shape, local_grid.tokens.shape)
print("channel pooled", eeg_features.tokens.shape)
print("channel weights", channel_weights.shape, channel_weights[0, 0])
assert not channel_weights[~local_grid.active.transpose(1, 2)].any()
ecg_encoder = backbone.modality_encoders["ecg"]
print("ECG temporal-only blocks", len(ecg_encoder.blocks))
assert not any("axis_attention" in n for n, _ in ecg_encoder.named_parameters())

## 5. Fusion → temporal readout
用 hooks 观察每一层输入，执行后立刻移除，避免后续 cell 重复打印。

In [ ]:
handles = []
for index, block in enumerate(backbone.fusion.blocks):
    handles.append(block.register_forward_pre_hook(
        lambda module, args, i=index: print("fusion block", i, tuple(args[0].shape))
    ))
try:
    output = backbone(batch, outputs=("patch_tokens", "local", "features", "joint"))
finally:
    for handle in handles:
        handle.remove()
print("fused temporal", output.joint.tokens.shape)
print("foundation", output.foundation_representation.shape, output.valid)
print("temporal weights", output.aux["temporal_weights"].shape)

## 6. LeJEPA loss 与梯度
global/local 在原始输入裁剪；同长度 views 合并一次 forward。所有 views 共用 backbone 和 projector。

In [ ]:
model.train()
views = view_sampler(batch)
print("view seconds", [v.groups["eeg"].values.shape[-1] // 200 for v in views])
model.zero_grad(set_to_none=True)
with autocast_context(config["training"], device):
    losses = model(views)
print("projections", losses.projections.shape)
print("loss/invariance/SIGReg", losses.loss.item(), losses.invariance.item(), losses.sigreg.item())
print("valid views", losses.valid)
losses.loss.backward()
print("CNN gradient norm", encoder.tokenizer.convolutions[0].weight.grad.norm().item())
print("projector gradient norm", model.projector[0].weight.grad.norm().item())

## 7. HF save/load
保存到 artifacts，不写回数据。加载环境需要安装本项目包。

In [ ]:
from backbone.huggingface import SleepWorldModel, SleepWorldModelConfig
from transformers import AutoModel

hf = SleepWorldModel(SleepWorldModelConfig(
    backbone_config=backbone.config.to_dict(), input_spec=backbone.input_spec
))
hf.backbone.load_state_dict(backbone.state_dict())
hf = hf.to(DEVICE).eval()
save_dir = ROOT / "artifacts/notebooks/component_hf"
hf.save_pretrained(save_dir)
restored = AutoModel.from_pretrained(save_dir, trust_remote_code=True).to(DEVICE).eval()
with torch.no_grad():
    expected = hf(batch=batch).pooler_output
    actual = restored(batch=batch).pooler_output
torch.testing.assert_close(expected, actual)
print("HF roundtrip", actual.shape, save_dir)